In [41]:
import pypdf, sentence_transformers, faiss
print("All libraries imported successfully")
print("pypdf version:", pypdf.__version__)

All libraries imported successfully
pypdf version: 6.14.2


Step 1: PDF text extraction and chunking

1. Extract text from the PDF

In [42]:
from pypdf import PdfReader

reader = PdfReader("retail.pdf")
pages_text = []

for i, page in enumerate(reader.pages):
    text = page.extract_text()
    pages_text.append(text)

print(f"Total pages: {len(pages_text)}")
print(pages_text[0][:500])  # preview first 500 chars of page 1

Total pages: 11



2. Clean up the text (optional but helps quality)



In [43]:
import re

def clean_text(text):
    text = re.sub(r'\s+', ' ', text)  # collapse whitespace/newlines
    text = text.strip()
    return text

cleaned_pages = [clean_text(t) for t in pages_text if t]
full_text = " ".join(cleaned_pages)
print("Total characters:", len(full_text))

Total characters: 20380


3. Chunk the text into overlapping segments

In [44]:
def chunk_text(text, chunk_size=800, overlap=100):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

chunks = chunk_text(full_text)
print(f"Total chunks created: {len(chunks)}")
print(chunks[0])

Total chunks created: 30
Retailers are rich in data but starving for actionable insights to drive top-line growth and bottom-line saving. Why are retailers struggling to leverage AI to drive growth? Siloed data is locked in legacy systems that are too slow to keep pace with the modern omnichannel shopper Dated analytics approaches leave money on the table and miss critical insights Black box algorithms and off-the-shelf solutions fail to fit unique business context and challenges Retailers rely on 1,000s of manual BI dashboards to try to uncover a needle in a haystack 02 Despite this, less than 10% of decisions are truly insights informed at the average retailer.* Retailers are sitting on mountains of data, but most are struggling with data stuck in legacy siloes and insights that can't keep up with double digit i


4. Save chunks to disk (so you don't re-parse every time)

In [45]:
import json

with open("chunks.json", "w") as f:
    json.dump(chunks, f)

print("Chunks saved to chunks.json")

Chunks saved to chunks.json


Step 2: Embeddings and FAISS retrieval index

1. Load the embedding model

In [47]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")  # small, fast, good quality
print("Model loaded. Embedding dimension:", model.get_sentence_embedding_dimension())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded. Embedding dimension: 384


/tmp/ipykernel_103166/3643089520.py:4: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Model loaded. Embedding dimension:", model.get_sentence_embedding_dimension())


2. Load your saved chunks (if starting fresh in this cell)

In [48]:
import json

with open("chunks.json", "r") as f:
    chunks = json.load(f)

print(f"Loaded {len(chunks)} chunks")

Loaded 30 chunks


3. Generate embeddings for all chunks

In [49]:
import numpy as np

embeddings = model.encode(chunks, show_progress_bar=True, convert_to_numpy=True)
print("Embeddings shape:", embeddings.shape)  # (num_chunks, embedding_dim)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings shape: (30, 384)


4. Build a FAISS index

In [50]:
import faiss

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)  # simple L2 distance index
index.add(embeddings.astype('float32'))

print("Total vectors in index:", index.ntotal)

Total vectors in index: 30


5. Save the index and embeddings to disk

In [51]:
faiss.write_index(index, "retail_index.faiss")
np.save("embeddings.npy", embeddings)

print("Index and embeddings saved")

Index and embeddings saved


6. Test retrieval with a sample question

In [52]:
def retrieve_top_k(query, k=3):
    query_embedding = model.encode([query], convert_to_numpy=True).astype('float32')
    distances, indices = index.search(query_embedding, k)
    return [chunks[i] for i in indices[0]]

results = retrieve_top_k("What does Tredence do for retailers?")
for i, r in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(r)
    print()

--- Result 1 ---
 are struggling with data stuck in legacy siloes and insights that can't keep up with double digit inflation, over $700B in on-hand inventory and recession level consumer sentiment.* 64% of retail insiders agreed that the use of AI to help businesses is more hype than reality. * Source: McKinsey | stlouisfed | Seeking Alpha | Wundermanthompson | Contentgrip | Retailwire AI-enabled retailers grow revenue 12.39% faster. $1T in lost sales due to out of stocks 72% of promotions fail to breakeven 30% of cloud spend is wasted 26% of the average marketing budget is wasted on inefficient campaigns Winning retailers have leapfrogged AI hype and put the true potential of AI/ML and now GenAI to work. Tredence combines deep data engineering and data science capabilities with retail expertise to help t

--- Result 2 ---
ur enterprise data into a profitable revenue stream Increases your marketing ROI Why Tredence? What do we do? Platform modernization Real-time actionable supply cha

Step 3: OpenAI integration

1. Install the OpenAI SDK

In [53]:
pip install openai python-dotenv

Note: you may need to restart the kernel to use updated packages.


2. Add your OpenAI key to .env

3. Configure the client in the notebook

In [54]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
print("OpenAI client ready")

OpenAI client ready


4. Replace ask_question to call OpenAI instead of Gemini

In [55]:
def ask_question(query, k=3):
    retrieved_chunks = retrieve_top_k(query, k=k)
    context = "\n\n".join(retrieved_chunks)

    prompt = f"""You are a helpful assistant answering questions based only on the provided document context.
If the answer isn't in the context, say you don't know.

Context:
{context}

Question: {query}

Answer:"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You answer questions using only the provided document context."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.2
    )
    return response.choices[0].message.content

5. Test it

In [56]:
answer = ask_question("What percentage of retail decisions are truly insights-informed?")
print(answer)

Less than 10% of decisions are truly insights informed at the average retailer.


In [57]:
answer2 = ask_question("What is Tredence's 7-step approach for data modernization?")
print(answer2)

The document does not provide specific details about Tredence's 7-step approach for data modernization.
